In [ ]:
#EXERCISE 1
import matplotlib.pyplot as plt
import numpy as np
from scipy import fft, fftpack

fs = 48000
num_samples = 480

def genSinus(f0, fs, num_samples):
    t = np.arange(num_samples) / fs
    sinusoid = np.sin(2 * np.pi * f0 * t)
    return sinusoid

signal_1 = genSinus(1000, fs, num_samples)
signal_2 = 0.3 * genSinus(15 * 1000, fs, num_samples)
signal_original = signal_1 + signal_2   # 1 kHz sine + 15 kHz sine

signal_fft = fft.fft(signal_original)   # DFT, complex coefficients
signal_fft_filtered = signal_fft.copy()
freqs = fftpack.fftfreq(len(signal_fft), d=1 / fs)   # frequency in Hz of each coefficient
signal_fft_filtered[np.abs(signal_fft) < 100] = 0    # small amplitudes (15 kHz) set to 0
filtered_time = fft.ifft(signal_fft_filtered).real   # back to time domain
t = np.arange(num_samples) / fs

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
ax1.stem(freqs, np.abs(signal_fft))
ax1.set_xlim(0, fs / 2)
ax2.stem(freqs, np.abs(signal_fft_filtered))
ax2.set_xlim(0, fs / 2)
plt.show()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
ax1.plot(t, signal_original)
ax2.plot(t, filtered_time)
plt.show()

In [ ]:
#EXERCISE 2
import matplotlib.pyplot as plt
import numpy as np
from scipy import fft, fftpack

fs = 48000
num_samples = 480

def genSinus(f0, fs, num_samples):
    t = np.arange(num_samples) / fs
    sinusoid = np.sin(2 * np.pi * f0 * t)
    return sinusoid

signal_1 = genSinus(1000, fs, num_samples)
signal_2 = 0.3 * genSinus(15 * 1000, fs, num_samples)
signal_original = signal_1 + signal_2

signal_fft = fft.fft(signal_original)
freqs = fftpack.fftfreq(len(signal_fft), d=1 / fs)
t = np.arange(num_samples) / fs

signal_fft_filtered = signal_fft.copy()
signal_fft_filtered[np.abs(signal_fft) > 100] = 0    # large amplitudes (1 kHz) set to 0
filtered_time = fft.ifft(signal_fft_filtered).real

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
ax1.stem(freqs, np.abs(signal_fft_filtered))
ax1.set_xlim(0, fs / 2)
ax2.plot(t, filtered_time)
plt.show()

In [ ]:
#EXERCISE 3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import fftpack
from scipy.signal.windows import gaussian


def fft_domaine_spectral(signal, filtre):
    fft_signal = fftpack.fft(signal)     # DFT of the signal
    fft_filtre = fftpack.fft(filtre)     # DFT of the filter
    return fftpack.ifft(fft_signal * fft_filtre)   # product in frequency = convolution in time

df = pd.read_csv('bitcoin1.csv')
x = df.loc[:, "medianTxValue(USD)"].to_numpy()

filtre = gaussian(M=31, std=25)   # gaussian window with 31 samples
filtre /= filtre.sum()            # sum 1, the mean value is kept

pad = np.zeros(x.size)            # filter padded with zeros to the length of the signal
pad[:filtre.shape[0]] = filtre

y = fft_domaine_spectral(x, pad)

M = len(filtre)
y_s = y[M-1:]                     # first M-1 samples are wrong (circular convolution)

t = np.arange(y_s.size)
plt.plot(t, y_s, "b")
plt.show()

In [ ]:
#EXERCISE 4
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import fftpack
from scipy.signal.windows import gaussian

df = pd.read_csv('bitcoin1.csv')

def fft_domaine_spectral(signal, filtre):
    N = len(signal) + len(filtre) - 1          # length of the full linear convolution
    fft_signal = fftpack.fft(signal, N)        # DFT with zero padding to N
    fft_filtre = fftpack.fft(filtre, N)
    return np.real(fftpack.ifft(fft_signal * fft_filtre))

x = df.loc[:, "price(USD)"].to_numpy()

filtre = gaussian(M=31, std=13)
filtre /= filtre.sum()

y = fft_domaine_spectral(x, filtre)

M = len(filtre)
delai = (M - 1) // 2                           # delay of the filter
y_s = y[delai:delai + len(x)]                  # keep only the part aligned with x

t = np.arange(x.size)

plt.plot(t, x, "r", label="Original signal")
plt.plot(t, y_s, "b", label="Filtered signal")
plt.xlabel("Sample")
plt.ylabel("Price [USD]")
plt.legend()
plt.grid()
plt.show()

In [ ]:
#EXERCISE 5
import matplotlib.pylab as plt
import numpy as np
from scipy import fftpack as fp


def rgb2gray(rgb):
    r, g, b = rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]
    gray = 0.2989 * r + 0.5870 * g + 0.1140 * b
    return gray

image = plt.imread("fruits.png")
img_grayscale = rgb2gray(image)

size = 25
kernel = np.zeros((size, size))
kernel[int((size - 1) / 2), :] = np.ones(size) # horizontal line, motion blur kernel
kernel = kernel / size

kernel1 = np.zeros([512, 512])                 # kernel put in the middle of an empty image
lim_g = int(np.floor(image.shape[0] / 2 - size / 2))
lim_d = int(np.floor(image.shape[0] / 2 + size / 2))
lim_h = int(np.floor(image.shape[1] / 2 - size / 2))
lim_b = int(np.floor(image.shape[1] / 2 + size / 2))
kernel1[lim_g:lim_d, lim_h:lim_b] = kernel

im_freq = fp.fft2(img_grayscale)               # DFT of the image
kernel_freq = fp.fft2(fp.ifftshift(kernel1))   # DFT of the kernel, H(u,v)

img_blurred_freq_domain = im_freq * kernel_freq    # convolution theorem
image_blurred_spatial_domain = fp.ifft2(img_blurred_freq_domain).real

# inverse filtering
epsilon = 1e-6
img_restored_freq_domain = img_blurred_freq_domain / (kernel_freq + epsilon)
image_restored = fp.ifft2(img_restored_freq_domain).real

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_grayscale, cmap=plt.get_cmap('gray'))
axes[0].set_title('Original image')
axes[1].imshow(image_blurred_spatial_domain, cmap=plt.get_cmap('gray'))
axes[1].set_title('Blurred image')
axes[2].imshow(image_restored, cmap=plt.get_cmap('gray'))
axes[2].set_title('Restored image')
for a in axes:
    a.axis('off')
plt.show()